# Final Working Pipeline

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [2]:
def generate_response(messages, max_new_tokens=1024):

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_ids = outputs[0][inputs.input_ids.shape[-1]:]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return response

In [3]:
sample_python = """
def merge_sort(arr):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2

    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    result = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])

    return result

print(merge_sort([7, 2, 9, 1, 5, 3]))
"""

In [4]:
import re

def extract_cpp(response):
    match = re.search(r"```(?:cpp|c\+\+)?\s*(.*?)```", response, re.DOTALL)

    if match:
        return match.group(1).strip()

    return response.strip()

In [5]:
import subprocess
import tempfile
import os

def compile_and_run(cpp_code):
    with tempfile.TemporaryDirectory() as tmpdir:
        cpp_file = os.path.join(tmpdir, "program.cpp")
        exe_file = os.path.join(tmpdir, "program")

        with open(cpp_file, "w") as f:
            f.write(cpp_code)

        compile_result = subprocess.run(
            ["g++", "-std=c++17", cpp_file, "-o", exe_file],
            capture_output=True,
            text=True
        )

        if compile_result.returncode != 0:
            return False, compile_result.stderr

        run_result = subprocess.run(
            [exe_file],
            capture_output=True,
            text=True,
            timeout=10
        )

        if run_result.returncode != 0:
            return False, run_result.stderr

        return True, run_result.stdout

In [6]:
def make_fix_prompt(python_code, cpp_code, error):

    return f"""
You are an expert Python-to-C++17 translator.

The following Python code must be translated into standalone C++17.

Python code:
{python_code}

Your previous C++ translation was:

{cpp_code}

However, the C++ code failed verification.

The compiler/runtime error was:

{error}

Fix the C++ code so that:
- It preserves the behavior of the Python code.
- It is complete standalone C++17.
- It includes all required headers.
- It contains a main() function.
- It compiles and runs correctly.

Return only the corrected C++ code.
"""

In [7]:
def translate_and_verify(python_code, max_retries=3):

    # Initial translation prompt
    prompt = f"""
Translate the following Python code into standalone C++17.

Requirements:
- Preserve the behavior of the Python code.
- Produce complete compilable C++17 code.
- Include all required headers.
- Include a main() function.
- Return only the C++ code.

Python code:

{python_code}
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert Python-to-C++ translator."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Try up to max_retries times
    for attempt in range(max_retries):

        print(f"\nAttempt {attempt + 1}/{max_retries}")

        # 1. Ask Qwen for C++
        response = generate_response(messages)

        # 2. Extract clean C++
        cpp_code = extract_cpp(response)

        # 3. Compile and execute it
        success, result = compile_and_run(cpp_code)

        # 4. If it works, we're done
        if success:
            print("✓ Compilation and execution successful!")
            return cpp_code

        # 5. If it failed, show the error
        print("✗ Verification failed.")
        print(result)

        # 6. Give the failed code + error back to Qwen
        fix_prompt = make_fix_prompt(
            python_code,
            cpp_code,
            result
        )

        messages = [
            {
                "role": "system",
                "content": "You are an expert Python-to-C++ translator."
            },
            {
                "role": "user",
                "content": fix_prompt
            }
        ]

    # All attempts failed
    raise RuntimeError(
        f"C++ translation failed after {max_retries} attempts."
    )

In [ ]:
if __name__ == "__main__":

    print("=== Python → C++ Translation Agent ===")

    final_cpp = translate_and_verify(
        sample_python,
        max_retries=3
    )

    print("\n=== Final Verified C++ ===")
    print(final_cpp)

=== Python → C++ Translation Agent ===

Attempt 1/3
